In [1]:
import pandas as pd

In [2]:
dir1 = r'C:\Users\praven.kumar\OneDrive - Cardinal Health\Desktop\SQL_Optimizer\Data'

dat = pd.read_csv(dir1 + r"\research_project_2.csv")

In [3]:
dat.columns

Index(['project_id', 'user_email', 'creation_date', 'total_tb_processed',
       'total_estimated_cost', 'query'],
      dtype='object')

In [4]:
import re

# --------------------------------------------------
# 1. Reference simple query (baseline for length)
# --------------------------------------------------
SIMPLE_QUERY = (
    "SELECT * FROM `edna-data-pr-cah.VI0_MED_MEDNA_DATA2_NP."
    "SALES_DISTRIBUTED_DATA_CV` LIMIT 1000"
)
MIN_QUERY_LENGTH = len(SIMPLE_QUERY)

# --------------------------------------------------
# 2. SQL complexity patterns
# --------------------------------------------------
COMPLEX_SQL_PATTERNS = [
    r"\bjoin\b",
    r"\bgroup\s+by\b",
    r"\bhaving\b",
    r"\bwith\b",
    r"\bwindow\b",
    r"\bover\s*\(",
    r"\bcase\s+when\b",
    r"\bsubquery\b",
    r"\bexists\b",
    r"\bunion\b",
    r"\bunnest\b",
    r"\barray\b",
    r"\bstruct\b"
]

# --------------------------------------------------
# 3. Helper functions
# --------------------------------------------------
def count_sql_keywords(query: str) -> int:
    query_lower = query.lower()
    return sum(bool(re.search(pattern, query_lower)) for pattern in COMPLEX_SQL_PATTERNS)

def has_subquery(query: str) -> bool:
    query_lower = query.lower()
    return query_lower.count("select") > 1

def count_tables(query: str) -> int:
    query_lower = query.lower()
    return len(re.findall(r"\bfrom\b|\bjoin\b", query_lower))

# --------------------------------------------------
# 4. Complexity scoring function
# --------------------------------------------------
def compute_complexity_score(query: str) -> int:
    score = 0

    # Length-based signal
    if len(query) >= MIN_QUERY_LENGTH:
        score += 1

    # SQL feature-based signals
    score += count_sql_keywords(query)

    if has_subquery(query):
        score += 2

    table_count = count_tables(query)
    if table_count >= 2:
        score += 1
    if table_count >= 4:
        score += 1

    return score

# --------------------------------------------------
# 5. Apply logic to dataframe
# --------------------------------------------------
dat["query_length"] = dat["query"].str.len()
dat["complexity_score"] = dat["query"].apply(compute_complexity_score)

# --------------------------------------------------
# 6. Filter complicated queries
# --------------------------------------------------
# You can tune this threshold
COMPLEXITY_THRESHOLD = 4

complicated_queries_df = dat[
    (dat["query_length"] >= MIN_QUERY_LENGTH) &
    (dat["complexity_score"] >= COMPLEXITY_THRESHOLD)
].copy()

# --------------------------------------------------
# 7. Optional: Sort by impact
# --------------------------------------------------
complicated_queries_df = complicated_queries_df.sort_values(
    by=["total_tb_processed", "complexity_score"],
    ascending=False
)


In [5]:
len(complicated_queries_df)

39

In [6]:
# Result dataframe
complicated_queries_df.head(10)

,project_id,user_email,creation_date,total_tb_processed,total_estimated_cost,query,query_length,complexity_score
5,edr-med-cmrclanlytc-pr-cah,rahul.sharma09@cardinalhealth.com,2026-02-09,2.43,12.15,select count(*) from (\r\nSELECT\r\n MONTH_DI...,71558,8
6,edr-med-cmrclanlytc-pr-cah,rahul.sharma09@cardinalhealth.com,2026-02-09,2.43,12.15,select count(*) from (\r\nSELECT\r\n MONTH_DI...,71689,8
7,edr-med-cmrclanlytc-pr-cah,shivam.deshmukh@cardinalhealth.com,2026-03-30,1.69,8.45,CREATE TABLE `edr-med-cmrclanlytc-pr-cah.SQL_Q...,25097,9
8,edr-med-cmrclanlytc-pr-cah,shivam.deshmukh@cardinalhealth.com,2026-03-30,1.68,8.40,CREATE TABLE `edr-med-cmrclanlytc-pr-cah.SQL_Q...,20710,9
14,edr-med-cmrclanlytc-pr-cah,andria.ullenbruch01@cardinalhealth.com,2025-12-05,1.29,6.45,"select ais.* -- distinct LEVEL5_NODE, Level5_E...",417,5
16,edr-med-cmrclanlytc-pr-cah,paul.grommesh@cardinalhealth.com,2025-12-11,1.26,6.30,"SELECT DISTINCT A.MATERIAL, A.STORAGECONDITI...",527,4
17,edr-med-cmrclanlytc-pr-cah,paul.grommesh@cardinalhealth.com,2025-12-11,1.26,6.30,"\r\n\r\n SELECT DISTINCT A.MATERIAL, A.STORAG...",519,4
21,edr-med-cmrclanlytc-pr-cah,paul.grommesh@cardinalhealth.com,2026-03-10,0.91,4.55,"with\r\nStorageCond as (\r\nSelect Y.Material,...",3711,9
20,edr-med-cmrclanlytc-pr-cah,anoop.pattanashetty@cardinalhealth.com,2026-03-27,0.91,4.55,"\r\nselect \r\n sum(DIST_REVENUE)\r\n ,sum(D...",469,5
28,edr-med-cmrclanlytc-pr-cah,anoop.pattanashetty@cardinalhealth.com,2026-03-27,0.69,3.45,"select year,month,sum(dist_tot_revenue) from (...",997,5


In [9]:
json_string = complicated_queries_df.to_json()

In [11]:
complicated_queries_df.to_json('research_project_2.json', orient='records', indent=4)